In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('data/raw',       exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

NULL_STRINGS = ['', 'NaN', 'nan', 'N/A', 'n/a',
                'Not Reported', 'not reported', 'None', '--', 'unknown']

## 1. Load Raw Data

In [2]:
json_file1 = r'C:\Users\user\Downloads\clinical.project-tcga-ov.2026-03-02.json'
json_file2 = r'C:\Users\user\Downloads\biospecimen.project-tcga-ov.2026-03-03.json'

df1 = pd.read_json(json_file1)
df2 = pd.read_json(json_file2)

df1.to_json('data/raw/clinical_raw.json',    orient='records')
df2.to_json('data/raw/biospecimen_raw.json', orient='records')

print('df1 shape:', df1.shape)
print('df2 shape:', df2.shape)

df1 shape: (608, 14)
df2 shape: (608, 4)


## 2. Inspect Nested Column Types (df1)

In [3]:
print('demographic type:', type(df1['demographic'][0]))
print('follow_ups type :', type(df1['follow_ups'][0]))
print('diagnoses type  :', type(df1['diagnoses'][0]))

demographic type: <class 'dict'>
follow_ups type : <class 'list'>
diagnoses type  : <class 'list'>


## 3. Flatten df1 (Clinical)

In [4]:
follow_ups_df = pd.json_normalize(df1['follow_ups'].explode()).reset_index(drop=True)
df1_clean = pd.concat([df1.drop(columns=['follow_ups']).reset_index(drop=True), follow_ups_df], axis=1)

diagnoses_df = pd.json_normalize(df1_clean['diagnoses'].explode()).reset_index(drop=True)
df1_clean = pd.concat([df1_clean.drop(columns=['diagnoses']).reset_index(drop=True), diagnoses_df], axis=1)

demographic_df = pd.json_normalize(df1_clean['demographic']).reset_index(drop=True)
df1_clean = pd.concat([df1_clean.drop(columns=['demographic']).reset_index(drop=True), demographic_df], axis=1)

df1_clean = df1_clean.loc[:, ~df1_clean.columns.duplicated()]
print('Shape after flatten:', df1_clean.shape)

Shape after flatten: (2582, 64)


In [5]:

dup_counts = df1_clean['case_id'].value_counts()
print('Max rows per case_id:', dup_counts.max())
print('Cases with >1 row   :', (dup_counts > 1).sum())

df1_clean = df1_clean.groupby('case_id', as_index=False).last()
print('Shape after dedup   :', df1_clean.shape)
print('Unique patients     :', df1_clean['case_id'].nunique())

Max rows per case_id: 1
Cases with >1 row   : 0
Shape after dedup   : (608, 64)
Unique patients     : 608


## 4. Clean df1

In [6]:
df1_clean.replace(NULL_STRINGS, np.nan, inplace=True)
print('Top 15 missing columns (%):')
print(df1_clean.isna().mean().mul(100).sort_values(ascending=False).head(15).round(1))

Top 15 missing columns (%):
figo_staging_edition_year             100.0
ajcc_pathologic_m                      99.8
ajcc_pathologic_t                      99.8
ajcc_pathologic_n                      99.8
ajcc_staging_system_edition            99.8
synchronous_malignancy                 99.3
prior_malignancy                       99.3
country_of_residence_at_enrollment     98.4
days_to_progression                    97.9
lost_to_followup                       97.2
evidence_of_progression_type           96.4
population_group                       96.1
karnofsky_performance_status           96.1
residual_disease                       94.9
ecog_performance_status                94.1
dtype: float64


In [7]:
keep_col = ['case_id','primary_site','disease_type','sex_at_birth','age_at_index',
            'days_to_birth','race','vital_status','days_to_death','figo_stage','tumor_grade']
df1_copy = df1_clean[keep_col].copy()


df1_copy['age_at_index'] = df1_copy['age_at_index'].fillna(df1_copy['days_to_birth'].abs() / 365.25)
df1_copy['age_at_index'] = df1_copy['age_at_index'].fillna(df1_copy['age_at_index'].median())


print('Outliers removed:', (~df1_copy['age_at_index'].between(18, 100)).sum())
df1_copy = df1_copy[df1_copy['age_at_index'].between(18, 100)].copy()


df1_copy['sex_at_birth'] = df1_copy['sex_at_birth'].fillna(df1_copy['sex_at_birth'].mode()[0])
df1_copy['disease_type'] = df1_copy['disease_type'].fillna(df1_copy['disease_type'].mode()[0])
df1_copy['race']         = df1_copy['race'].fillna('Unknown')
df1_copy['figo_stage']   = df1_copy['figo_stage'].fillna('Unknown')
df1_copy['tumor_grade']  = df1_copy['tumor_grade'].fillna('Unknown')


df1_copy.loc[df1_copy['vital_status'].isna() & df1_copy['days_to_death'].notna(), 'vital_status'] = 'Dead'
df1_copy.loc[df1_copy['vital_status'].isna() & df1_copy['days_to_death'].isna(),  'vital_status'] = 'Unknown/Censored'
df1_copy['vital_status'] = df1_copy['vital_status'].fillna('Unknown/Censored')

df1_copy['days_to_birth'] = -(df1_copy['age_at_index'] * 365.25).round()

print('Nulls in df1_copy:')
print(df1_copy.isna().sum())

Outliers removed: 0
Nulls in df1_copy:
case_id            0
primary_site       0
disease_type       0
sex_at_birth       0
age_at_index       0
days_to_birth      0
race               0
vital_status       0
days_to_death    260
figo_stage         0
tumor_grade        0
dtype: int64


## 5. Flatten & Clean df2 (Biospecimen)

In [8]:
samples_df = pd.json_normalize(df2['samples'].explode()).reset_index(drop=True)
df2_clean = pd.concat([df2.drop(columns=['samples']).reset_index(drop=True), samples_df], axis=1)

portions_df = pd.json_normalize(df2_clean['portions'].explode()).reset_index(drop=True)
df2_clean = pd.concat([df2_clean.drop(columns=['portions']).reset_index(drop=True), portions_df], axis=1)

df2_clean = df2_clean.loc[:, ~df2_clean.columns.duplicated()]


dup_counts2 = df2_clean['case_id'].value_counts()
print('Max rows per case_id:', dup_counts2.max())
print('Unique patients     :', df2_clean['case_id'].nunique())
print('Shape               :', df2_clean.shape)

Max rows per case_id: 1
Unique patients     : 608
Shape               : (2430, 33)


In [9]:
if 'analytes' in df2_clean.columns:
    df2_clean['analyte_type'] = df2_clean['analytes'].apply(
        lambda x: x[0].get('analyte_type') if isinstance(x, list) and len(x) > 0 else np.nan
    )
    df2_clean = df2_clean.drop(columns=['analytes'])
    print('analyte_type values:')
    print(df2_clean['analyte_type'].value_counts())

analyte_type values:
analyte_type
DNA                                   361
Repli-G (Qiagen) DNA                  326
GenomePlex (Rubicon) Amplified DNA    266
RNA                                   145
Total RNA                             127
Name: count, dtype: int64


In [10]:
keep_cols = ['case_id','submitter_id','tumor_descriptor','sample_id',
             'sample_type','tissue_type','weight','is_ffpe','analyte_type']
df2_clean = df2_clean[keep_cols]
df2_clean = df2_clean.dropna(subset=['case_id'])
df2_clean.replace(['nan','None','Not Reported',''], np.nan, inplace=True)


median_weight = df2_clean.groupby('sample_type')['weight'].transform('median')
df2_clean['weight'] = df2_clean['weight'].fillna(median_weight)
df2_clean['weight'] = df2_clean['weight'].fillna(df2_clean['weight'].median())

df2_clean['is_ffpe'] = df2_clean['is_ffpe'].fillna(df2_clean['is_ffpe'].mode()[0])

print('Nulls in df2_clean:')
print(df2_clean.isna().sum())

Nulls in df2_clean:
case_id               0
submitter_id          0
tumor_descriptor      0
sample_id             0
sample_type           0
tissue_type           0
weight                0
is_ffpe               0
analyte_type        286
dtype: int64


In [11]:
df1_copy.to_csv('data/processed/clinical_clean.csv',     index=False)
df2_clean.to_csv('data/processed/biospecimen_clean.csv', index=False)
print('Saved: data/processed/clinical_clean.csv    ', df1_copy.shape)
print('Saved: data/processed/biospecimen_clean.csv ', df2_clean.shape)

Saved: data/processed/clinical_clean.csv     (608, 11)
Saved: data/processed/biospecimen_clean.csv  (608, 9)
